# Phân vùng mặt đường car6: TwinLiteNetPlus, hai chế độ `compare` và `final`
Notebook huấn luyện phân vùng hai lớp `background` (nền, 0) và `road` (mặt đường, 1) cho xe car6 trên Colab GPU. Khung chạy giống `yolo_train_car.ipynb`: **một ô cấu hình duy nhất** (ô 2), chọn `MODE`, bấm Runtime → Run all.

- **Hợp đồng tensor (cố định):** đầu vào `input` `[1,3,192,320]` float32 NCHW, RGB, chia 255 rồi chuẩn hóa ImageNet ngoài mạng; đầu ra `output` `[1,2,192,320]` logits (điểm chưa chuẩn hóa); đường khi kênh 1 > kênh 0; ONNX opset 12. Hợp đồng lấy từ `models/twinlitenetplus_192x320_road_info.json`; dữ liệu xem `docs/datasets.md`.
- **Dữ liệu:** `ucr_road.zip` (ảnh + nhãn COCO polygon) và `split.csv` chỉ có **hai tập `train`, `valid`** (giống bộ YOLO `ucr_det`, không có tập test). Notebook chia theo cột `split` của `split.csv` (chia theo lượt thu, cột `group_id`), **không** theo thư mục trong zip.
- **Cách dùng nhanh:** đặt `ucr_road.zip` và `split.csv` vào `MyDrive/UIT_Car_Racing/Vong3/data/`, đặt `MODE="compare"`, Runtime → GPU → Run all. Cuối cùng notebook tải `results.zip`.

Hai chế độ `MODE`:
1. `"compare"`: mỗi biến thể trong `VARIANTS` × mỗi seed trong `SEEDS`: train trên `train`, chọn epoch có mIoU `valid` cao nhất, đánh giá `valid` ở 5 mức sáng, ghi thêm dòng vào `Vong3/compare/twinlite_compare.csv`, xuất ONNX cho seed đầu của mỗi biến thể. Chạy lại sau khi Colab ngắt: lần chạy đã có trong CSV được bỏ qua, trọng số đã train (`Vong3/twinlite/weights/<DATA_TAG>/`) được dùng lại. **Chế độ này cho ra model kèm số đo `valid` để ghi vào README.**
2. `"final"`: FINAL_FIT với `FINAL_VARIANT`, train trên **toàn bộ ảnh có nhãn (`train` + `valid`)**, lấy epoch cuối, xuất ONNX. Không còn tập nào để đo, nên **không có chỉ số** (đừng ghi số nào cho model này).

Chỉ số `valid` chỉ dùng để lọc thô; quyết định cuối dựa trên chạy thật trên sa bàn. `FAKE_DATA=True` kiểm quy trình trên CPU bằng ảnh giả, không cần Drive hay GPU.

## 2. Môi trường và cấu hình duy nhất
Chọn Runtime → Change runtime type → GPU. `TWINLITE_COMMIT` ghim commit của repo nguồn có API `SingleLiteNetPlus` dùng ở mục 4. `DATA_ZIP`, `SPLIT_CSV` tính từ `Vong3/` trên Drive (nếu không thấy `DATA_ZIP` mà `Vong3/data` có đúng một zip có chữ `road` trong tên thì dùng zip đó). `REF_DIR` tùy chọn, chứa các cặp tensor tham chiếu cũ `NNN_input.npy`, `NNN_logits.npy`.

Mọi siêu tham số nằm trong ô này. Không đổi kích thước, chuẩn hóa hoặc opset (nằm ở `CONTRACT`) nếu cần giữ hợp đồng suy luận. Notebook dùng torch/numpy/OpenCV/matplotlib có sẵn trên Colab (không ghim, vì ghim cũ không cài được trên Python 3.13), chỉ cài onnx và onnxruntime nếu thiếu; phiên bản thực tế được ghi vào `twinlite_metrics.json`. Cần Internet để cài gói và clone repo nguồn: https://github.com/chequanghuy/TwinLiteNetPlus ; [LICENSE upstream](https://github.com/chequanghuy/TwinLiteNetPlus/blob/main/LICENSE) ghi MIT License. Ô thiết lập in LICENSE tại chính commit đã chọn để đối chiếu.

In [ ]:
MODE = "compare"          # "compare" | "final"; xem ô mô tả đầu notebook.
FAKE_DATA = False         # True: ảnh giả trên CPU để kiểm quy trình (không cần Drive, GPU).
DATA_TAG = "ucr_road"     # Định danh bộ dữ liệu, dùng trong tên thư mục/khóa lần chạy.
DATA_ZIP = "data/ucr_road.zip"   # Zip COCO polygon, tính từ Vong3/ trên Drive.
SPLIT_CSV = "data/split.csv"     # Bảng chia tập (cột file, split, group_id); split chỉ nhận train, valid.
VARIANTS = ["small"]      # Biến thể TwinLiteNetPlus: nano | small | medium | large. Model đang dùng cho xe: small.
SEEDS = [42]              # Có thể [42, 43] để đo độ lệch giữa seed; lần chạy có trong CSV được bỏ qua.
EPOCHS = 100
BATCH = 16
LR = 1e-3                 # AdamW, learning rate cố định
WEIGHT_DECAY = 1e-4
FLIP_PROBABILITY = 0.5    # Tăng cường train: lật ngang ảnh+mask
BRIGHTNESS_JITTER = 0.25  # Tăng cường train: cộng độ sáng ±25% (của 255)
CONTRAST_JITTER = 0.20    # Tăng cường train: nhân tương phản ×(1±0,2)
WORKERS = 2
AMP = True                # Tính toán hỗn hợp độ chính xác; tự tắt khi chạy CPU
BRIGHTNESS = [0.6, 0.8, 1.0, 1.2, 1.4]   # Compare: đánh giá valid ở các mức sáng này
EXPORT_FIRST_SEED = True  # Compare: xuất ONNX cho seed đầu của mỗi biến thể
FINAL_VARIANT = "small"   # Final: biến thể train trên train+valid
FINAL_SEED = 42
ONNX_CHECK_N = 100        # Số ảnh (trải đều) dùng đối chiếu ONNX Runtime với PyTorch
PREVIEW_N = 6             # Số ảnh valid vẽ ảnh | nhãn | dự đoán (0 = tắt)
REF_DIR = ""              # Tùy chọn: thư mục tensor tham chiếu cũ; để trống nếu không có.
TWINLITE_COMMIT = "90f1b8695ae311d5123b05f8534b2e11e42499d2"  # commit ghim của repo nguồn
EXTRA_PACKAGES = ["onnx>=1.17", "onnxruntime>=1.20"]
INSTALL_DEPS = True
LOCAL_ROOT = "./MyDrive/UIT_Car_Racing/Vong3"   # Chạy ngoài Colab: thay cho Drive
WORK_DIR = "/content/work_twinlite"             # Đĩa Colab

# Hợp đồng suy luận với C++ trên xe: KHÔNG đổi.
CONTRACT = dict(height=192, width=320, classes=2, opset=12, scale=1 / 255,
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], ort_atol=1e-4, ort_rtol=1e-3)

assert MODE in ("compare", "final"), "MODE chỉ nhận compare hoặc final"
assert (CONTRACT["height"], CONTRACT["width"], CONTRACT["classes"], CONTRACT["opset"]) == (192, 320, 2, 12), "Giữ hợp đồng tensor cố định"
assert EPOCHS > 0 and BATCH > 1 and LR > 0 and 0 <= FLIP_PROBABILITY <= 1 and WORKERS >= 0
assert set(VARIANTS) | {FINAL_VARIANT} <= {"nano", "small", "medium", "large"}

## Môi trường, thư mục và hàm dùng chung
Kết quả ở `MyDrive/UIT_Car_Racing/Vong3/twinlite/`; bảng so sánh ở `Vong3/compare/twinlite_compare.csv`; trọng số ở `Vong3/twinlite/weights/<DATA_TAG>/<biến thể>_s<seed>[_final].pt` cùng file `.json` lịch sử; dữ liệu giải nén và mã nguồn clone nằm trên đĩa Colab. Mỗi lần chạy tạo thư mục `Vong3/twinlite/real/<RUN_ID>/<biến thể>_s<seed>[_final]/` chứa `best.pt` (hoặc `last.pt` khi final), `twinlitenetplus_192x320.onnx`, `twinlitenetplus_192x320_road_info.json`, `twinlite_metrics.json` và biểu đồ; `results.zip` là cả thư mục `<RUN_ID>` đó.

In [ ]:
import os, sys, subprocess, importlib.util, re, json, csv, zipfile, random, hashlib, shutil, time, datetime
from pathlib import Path
from collections import defaultdict
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
if INSTALL_DEPS:
    needed = {"onnx": EXTRA_PACKAGES[0], "onnxruntime": EXTRA_PACKAGES[1]}
    missing = [req for mod, req in needed.items() if importlib.util.find_spec(mod) is None]
    if missing: subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True, timeout=900)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if not FAKE_DATA: assert DEVICE == "cuda", "Cần Colab GPU và torch CUDA hoạt động (hoặc đặt FAKE_DATA=True để kiểm trên CPU)"
if DEVICE == "cuda": subprocess.run(["nvidia-smi"], check=True, timeout=30)
USE_AMP = bool(AMP and DEVICE == "cuda")
import numpy, cv2, matplotlib, onnx, onnxruntime
if not IN_COLAB: matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
PACKAGES = dict(python=sys.version.split()[0], torch=torch.__version__, numpy=numpy.__version__, opencv=cv2.__version__,
                matplotlib=matplotlib.__version__, onnx=onnx.__version__, onnxruntime=onnxruntime.__version__)
print("Thiết bị:", DEVICE, "| phiên bản:", PACKAGES)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/UIT_Car_Racing/Vong3")
    WORK = Path(WORK_DIR)
else:
    ROOT = Path(LOCAL_ROOT).expanduser().resolve()
    WORK = ROOT / "work_twinlite"
ROOT.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
KIND = "fake" if FAKE_DATA else "real"
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "twinlite" / KIND / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
WEIGHTS_DIR = ROOT / "twinlite" / "weights" / (DATA_TAG + ("_fake" if FAKE_DATA else ""))
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
COMPARE_CSV = ROOT / "compare" / ("twinlite_compare.csv" if not FAKE_DATA else "twinlite_compare_fake.csv")
COMPARE_CSV.parent.mkdir(parents=True, exist_ok=True)

assert re.fullmatch(r"[0-9a-fA-F]{40}", TWINLITE_COMMIT), "Điền TWINLITE_COMMIT: hash commit 40 ký tự của repo nguồn"
REPO = WORK / "TwinLiteNetPlus"
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/chequanghuy/TwinLiteNetPlus.git", str(REPO)], check=True, timeout=300)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", TWINLITE_COMMIT], check=True, timeout=60)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit.lower() == TWINLITE_COMMIT.lower()
assert (REPO / "LICENSE").is_file(), "Đọc và đối chiếu giấy phép repo nguồn trước khi tiếp tục"
print((REPO / "LICENSE").read_text()[:400], "...")
sys.path.insert(0, str(REPO))

def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

def json_safe(value):
    if isinstance(value, dict): return {k: json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_safe(v) for v in value]
    if isinstance(value, float) and not np.isfinite(value): return None
    return value

CSV_COLUMNS = ["run_key", "run_id", "data_tag", "variant", "seed", "epochs", "batch_size", "lr", "params", "train_min",
               "n_train", "n_valid", "best_epoch", "brightness", "iou_background", "iou_road", "miou", "precision_road",
               "recall_road", "weights", "split_sha256", "upstream_commit"]

def done_keys(path):
    if not path.exists(): return set()
    with path.open(encoding="utf-8-sig", newline="") as f: return {r["run_key"] for r in csv.DictReader(f)}

def append_csv(path, rows):
    new = not path.exists()
    if not new:
        with path.open(encoding="utf-8-sig", newline="") as f: header = next(csv.reader(f))
        assert header == CSV_COLUMNS, "Cột CSV cũ khác bản này; đổi tên file CSV cũ rồi chạy lại: " + str(path)
    with path.open("a", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, CSV_COLUMNS)
        if new: w.writeheader()
        w.writerows(rows)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE == "cuda": torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 3. Dữ liệu COCO polygon và `split.csv`
Zip chứa ảnh và nhãn COCO polygon (`_annotations.coco.json` trong từng thư mục). Notebook **không** dùng cách chia thư mục của zip: nó gom ảnh của mọi thư mục rồi chia lại đúng theo cột `split` trong `split.csv` (chỉ `train` hoặc `valid`; chia theo lượt thu, không còn lượt nào nằm ở nhiều tập). Tên ảnh trong cột `file` phải trùng `file_name` trong JSON; thiếu, trùng hoặc có nhóm lượt thu (`group_id`) nằm ở nhiều tập thì notebook dừng. Category gốc ID 0 là lớp cha, không có annotation; không được hiểu ID 0 là pixel nền. Mọi annotation thuộc category không mang tên nền được tô road=1; nền là phần ảnh không được polygon phủ. Ô đọc dừng nếu ID 0 có annotation để người dùng kiểm nhãn.

Polygon (đa giác) được tô ở kích thước ảnh gốc rồi resize mask bằng nearest-neighbor (láng giềng gần nhất). Ảnh được resize toàn bộ bằng INTER_LINEAR, không crop hoặc padding; đổi BGR→RGB, nhân 1/255 rồi chuẩn hóa ImageNet ngoài model. Tăng cường chỉ áp dụng khi train: lật ngang đồng thời ảnh/mask và đổi độ sáng/tương phản ảnh. `FAKE_DATA=True`: tạo ảnh 1280×720 giả có hình thang "đường" cùng `split.csv` giả.

In [ ]:
def fake_dataset(folder, n=40):
    """Ảnh giả + COCO polygon + split.csv cùng dạng bộ thật (2 thư mục train/valid, category 0 là lớp cha)."""
    rng = np.random.default_rng(0)
    parts = ("train", "valid")
    coco = {p: dict(images=[], annotations=[], categories=[dict(id=0, name="road-parent"), dict(id=1, name="road")]) for p in parts}
    for p in parts: (folder / p).mkdir(parents=True)
    rows, ann_id = [], 1
    for i in range(n):
        g = i // 4; part = "valid" if g in (8, 9) else "train"
        name = "fake_p{:03d}_{:02d}.jpg".format(g, i % 4)
        img = np.full((720, 1280, 3), (40, 120, 40), np.uint8)
        y0 = int(rng.integers(300, 420)); xl, xr = int(rng.integers(300, 500)), int(rng.integers(780, 980))
        pts = [[xl, y0], [xr, y0], [1200, 719], [80, 719]]
        cv2.fillPoly(img, [np.array(pts, np.int32)], (90, 90, 90))
        img = np.clip(img.astype(np.float32) + rng.normal(0, 6, img.shape), 0, 255).astype(np.uint8)
        cv2.imwrite(str(folder / part / name), img)
        c = coco[part]; iid = len(c["images"]) + 1
        c["images"].append(dict(id=iid, file_name=name, width=1280, height=720))
        c["annotations"].append(dict(id=ann_id, image_id=iid, category_id=1, segmentation=[sum(pts, [])], iscrowd=0)); ann_id += 1
        rows.append([name, part, "g{:03d}".format(g), 1280, 720])
    for p in parts: (folder / p / "_annotations.coco.json").write_text(json.dumps(coco[p]), encoding="utf-8")
    with (folder / "split.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["file", "split", "group_id", "width", "height"]); w.writerows(rows)

DATA_ROOT = WORK / ("road_fake" if FAKE_DATA else "road_dataset")
if FAKE_DATA:
    if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
    fake_dataset(DATA_ROOT)
    zip_path, split_path = Path("fake.zip"), DATA_ROOT / "split.csv"
else:
    zip_path = ROOT / DATA_ZIP
    if not zip_path.is_file():
        found = [p for p in (ROOT / "data").glob("*.zip") if "road" in p.name.lower()]
        assert len(found) == 1, "Không thấy {}. Các zip trong Vong3/data: {}".format(DATA_ZIP, [p.name for p in (ROOT / "data").glob("*.zip")])
        zip_path = found[0]
    split_path = ROOT / SPLIT_CSV
    assert split_path.is_file(), "Không thấy {}. Tải split.csv lên cùng thư mục với zip.".format(split_path)
    print("Zip:", zip_path.name, "|", zip_path.stat().st_size, "byte | split:", split_path.name)
    if not (DATA_ROOT / ".unzipped").exists():
        if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
        local_zip = WORK / zip_path.name
        shutil.copyfile(zip_path, local_zip)
        with zipfile.ZipFile(local_zip) as archive:
            for item in archive.infolist():
                target = (DATA_ROOT / item.filename.replace("\\", "/")).resolve()
                assert target.is_relative_to(DATA_ROOT.resolve()), "Đường dẫn zip vượt thư mục dữ liệu"
                assert (item.external_attr >> 16) & 0o170000 != 0o120000, "Không nhận symlink trong zip"
            archive.extractall(DATA_ROOT)
        (DATA_ROOT / ".unzipped").write_text("ok")
    else:
        print("Dùng lại dữ liệu đã giải nén:", DATA_ROOT)

In [ ]:
def polygon_to_mask(height, width, annotations, categories):
    mask = np.zeros((height, width), dtype=np.uint8)
    names = {int(c["id"]): c["name"].strip().lower() for c in categories}
    for ann in annotations:
        cid = int(ann["category_id"])
        assert cid in names, "Category không khai báo"
        if names[cid] in ("background", "nền"):
            continue
        polygons = ann["segmentation"]
        assert isinstance(polygons, list) and polygons, "Chỉ nhận COCO polygon, không nhận RLE"
        for polygon in polygons:
            points = np.asarray(polygon, dtype=np.float64)
            assert points.size >= 6 and points.size % 2 == 0 and np.isfinite(points).all(), "Polygon không hợp lệ"
            points = np.rint(points.reshape(-1, 2)).clip(-1e6, 1e6).astype(np.int32)
            cv2.fillPoly(mask, [points], 1)
    return mask

def preprocess_bgr(bgr):
    assert bgr is not None and bgr.ndim == 3 and bgr.shape[2] == 3
    small = cv2.resize(bgr, (CONTRACT["width"], CONTRACT["height"]), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
    x = rgb.astype(np.float32) * np.float32(CONTRACT["scale"])
    x = (x - np.asarray(CONTRACT["mean"], np.float32)) / np.asarray(CONTRACT["std"], np.float32)
    return np.ascontiguousarray(x.transpose(2, 0, 1), dtype=np.float32)

from torch.utils.data import Dataset, DataLoader

def load_split_csv(path):
    with open(path, encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert rows, "split.csv rỗng"
    for column in ("file", "split"):
        assert column in rows[0], "split.csv thiếu cột " + column
    names = [r["file"] for r in rows]
    assert len(set(names)) == len(names), "split.csv có tên file trùng"
    bad = {r["split"] for r in rows} - {"train", "valid"}
    assert not bad, "Giá trị split không hợp lệ (chỉ train, valid): " + str(bad)
    if "group_id" in rows[0]:
        seen = defaultdict(set)
        for r in rows:
            if r["group_id"]: seen[r["group_id"]].add(r["split"])
        leaks = [g for g, s in seen.items() if len(s) > 1]
        assert not leaks, "{} nhóm lượt thu nằm ở nhiều tập: {}".format(len(leaks), leaks[:5])
    return rows

def build_index(root):
    index, categories, key0 = {}, None, None
    json_files = sorted(root.rglob("_annotations.coco.json"))
    assert json_files, "Zip không có _annotations.coco.json"
    for json_path in json_files:
        coco = json.loads(json_path.read_text(encoding="utf-8"))
        key = [(int(c["id"]), c["name"]) for c in coco["categories"]]
        if key0 is None: key0, categories = key, coco["categories"]
        assert key == key0, "Các JSON có categories khác nhau"
        ids = {im["id"] for im in coco["images"]}
        assert len(ids) == len(coco["images"]) and ids, "ID ảnh trùng hoặc JSON rỗng: " + str(json_path)
        annotations = defaultdict(list)
        for ann in coco["annotations"]:
            assert ann["image_id"] in ids
            assert int(ann["category_id"]) != 0, "Category cha ID 0 có annotation: kiểm lại nhãn"
            annotations[ann["image_id"]].append(ann)
        for im in coco["images"]:
            assert im["file_name"] not in index, "Ảnh trùng tên giữa các thư mục: " + im["file_name"]
            index[im["file_name"]] = dict(root=json_path.parent, im=im, anns=annotations[im["id"]])
    return index, categories

SPLIT_ROWS = load_split_csv(split_path)
INDEX, CATEGORIES = build_index(DATA_ROOT)
missing = [r["file"] for r in SPLIT_ROWS if r["file"] not in INDEX]
assert not missing, "{} file trong split.csv không có trong zip, ví dụ: {}".format(len(missing), missing[:3])
for r in SPLIT_ROWS:
    im = INDEX[r["file"]]["im"]
    if r.get("width") and r.get("height"):
        assert (int(r["width"]), int(r["height"])) == (im["width"], im["height"]), "Kích thước khác split.csv: " + r["file"]
unused = len(INDEX) - len(SPLIT_ROWS)
if unused: print("Cảnh báo:", unused, "ảnh trong zip không có trong split.csv, bỏ qua")
SPLIT_COUNTS = {s: sum(r["split"] == s for r in SPLIT_ROWS) for s in ("train", "valid")}
print("Tổng ảnh theo split.csv:", SPLIT_COUNTS)
SPLIT_SHA256 = hashlib.sha256(split_path.read_bytes()).hexdigest()

class RoadDataset(Dataset):
    def __init__(self, splits, augment=False, brightness=1.0):
        self.records = [INDEX[r["file"]] for r in SPLIT_ROWS if r["split"] in splits]
        assert self.records, "Split rỗng: " + str(splits)
        self.categories, self.augment, self.brightness = CATEGORIES, augment, brightness
    def __len__(self):
        return len(self.records)
    def raw(self, index):
        rec = self.records[index]; im = rec["im"]
        path = (rec["root"] / im["file_name"]).resolve()
        assert path.is_relative_to(rec["root"].resolve()), "Ảnh vượt thư mục dữ liệu"
        bgr = cv2.imread(str(path))
        assert bgr is not None, "Không đọc được ảnh: " + str(path)
        assert bgr.shape[:2] == (im["height"], im["width"]), "Kích thước ảnh khác JSON"
        return bgr, polygon_to_mask(im["height"], im["width"], rec["anns"], self.categories)
    def __getitem__(self, index):
        bgr, mask = self.raw(index)
        if self.augment:
            if random.random() < FLIP_PROBABILITY: bgr, mask = cv2.flip(bgr, 1), cv2.flip(mask, 1)
            alpha = random.uniform(1 - CONTRAST_JITTER, 1 + CONTRAST_JITTER)
            beta = random.uniform(-BRIGHTNESS_JITTER, BRIGHTNESS_JITTER) * 255
            bgr = np.clip(bgr.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        if self.brightness != 1.0:
            bgr = np.clip(bgr.astype(np.float32) * self.brightness, 0, 255).astype(np.uint8)
        x = preprocess_bgr(bgr)
        y = cv2.resize(mask, (CONTRACT["width"], CONTRACT["height"]), interpolation=cv2.INTER_NEAREST)
        return torch.from_numpy(x), torch.from_numpy(y.astype(np.int64))

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed); random.seed(worker_seed)

def make_loader(dataset, shuffle, seed):
    return DataLoader(dataset, batch_size=BATCH, shuffle=shuffle, num_workers=WORKERS, pin_memory=DEVICE == "cuda",
                      worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(seed))

preview_ds = RoadDataset(("train",))
fig, axes = plt.subplots(4, 2, figsize=(10, 12))
for i in range(4):
    bgr, mask = preview_ds.raw(i % len(preview_ds))
    axes[i, 0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); axes[i, 1].imshow(mask, vmin=0, vmax=1, cmap="gray")
    for ax in axes[i]: ax.axis("off")
fig.tight_layout(); fig.savefig(OUT / "data_preview.png"); plt.show()

## 4. Mô hình
Dò chữ ký tại commit đã clone, rồi dùng `SingleLiteNetPlus(args.config)` của repo nguồn: nhánh phân vùng đơn của họ TwinLiteNetPlus (biến thể `nano`, `small`, `medium`, `large`; `small` là biến thể dùng cho xe). Không dùng loss hoặc dataloader upstream. Kiểm tra tensor đầu ra trước khi train; dừng nếu API hoặc hình dạng khác. Không thêm softmax vào model; argmax trực tiếp trên hai kênh, hòa chọn nền.

In [ ]:
import inspect
from types import SimpleNamespace
from model.model import SingleLiteNetPlus
from model import config as upstream_config
print("Chữ ký:", inspect.signature(SingleLiteNetPlus))
print("Biến thể upstream:", list(upstream_config.sc_ch_dict))

def build_model(variant, device):
    assert variant in upstream_config.sc_ch_dict, "Biến thể lạ: " + variant
    return SingleLiteNetPlus(SimpleNamespace(config=variant)).to(device)

for _v in sorted(set(VARIANTS) | {FINAL_VARIANT}):
    _m = build_model(_v, DEVICE).eval()
    with torch.no_grad(): _p = _m(torch.zeros(1, 3, CONTRACT["height"], CONTRACT["width"], device=DEVICE))
    assert isinstance(_p, torch.Tensor) and tuple(_p.shape) == (1, 2, CONTRACT["height"], CONTRACT["width"])
    assert torch.isfinite(_p).all()
    print(_v, "| cấu hình", upstream_config.sc_ch_dict[_v], "| tham số", sum(p.numel() for p in _m.parameters()))
del _m, _p

## 5. Huấn luyện
AMP giảm bộ nhớ GPU. CrossEntropyLoss và AdamW với learning rate cố định. `compare`: train trên `train`, lưu checkpoint có **mIoU `valid` cao nhất** (`best.pt`). `final`: train trên `train`+`valid`, lấy **epoch cuối** (`last.pt`), không đánh giá. mIoU là trung bình IoU các lớp có hợp khác 0; chỉ số không xác định ghi null khi xuất JSON. Trọng số đã có ở `Vong3/twinlite/weights/<DATA_TAG>/` thì dùng lại, không train lại (muốn train lại: xóa file `.pt` và `.json` tương ứng).

In [ ]:
criterion = torch.nn.CrossEntropyLoss()

def scores(matrix):
    m = matrix.astype(np.float64); tp = np.diag(m)
    union = m.sum(0) + m.sum(1) - tp
    iou = np.divide(tp, union, out=np.full(2, np.nan), where=union > 0)
    precision = tp[1] / m[:, 1].sum() if m[:, 1].sum() else float("nan")
    recall = tp[1] / m[1].sum() if m[1].sum() else float("nan")
    return dict(iou_background=float(iou[0]), iou_road=float(iou[1]), miou=float(np.nanmean(iou)),
                precision_road=float(precision), recall_road=float(recall))

def run_epoch(model, loader, training, optimizer=None, scaler=None, amp=False):
    model.train(training)
    total_loss, count = 0.0, 0
    matrix = np.zeros((2, 2), np.int64)
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            if training: optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=amp):
                logits = model(x)
                loss = criterion(logits, y)
            assert torch.isfinite(loss), "Loss không hữu hạn"
            if training:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total_loss += loss.item() * len(x); count += len(x)
            indices = (y * 2 + logits.detach().argmax(1)).reshape(-1)
            matrix += torch.bincount(indices, minlength=4).reshape(2, 2).cpu().numpy()
    return dict(loss=total_loss / count, **scores(matrix)), matrix

def run_config(variant, seed, final):
    return dict(variant=variant, seed=seed, final_fit=final, trained_on="train+valid" if final else "train", epochs=EPOCHS,
                batch_size=BATCH, lr=LR, weight_decay=WEIGHT_DECAY, optimizer="AdamW", loss="CrossEntropyLoss",
                flip_probability=FLIP_PROBABILITY, brightness_jitter=BRIGHTNESS_JITTER, contrast_jitter=CONTRAST_JITTER,
                amp=USE_AMP, workers=WORKERS, **CONTRACT)

def train_or_load(variant, seed, final):
    tkey = "{}_s{}{}".format(variant, seed, "_final" if final else "")
    wpath = WEIGHTS_DIR / (tkey + ".pt")
    if wpath.exists():
        print("  Dùng lại trọng số:", wpath.name); return wpath, None
    set_seed(seed); t0 = time.time()
    model = build_model(variant, DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    train_loader = make_loader(RoadDataset(("train", "valid") if final else ("train",), augment=True), True, seed)
    valid_loader = None if final else make_loader(RoadDataset(("valid",)), False, seed)
    history, best = [], -1.0
    for epoch in range(1, EPOCHS + 1):
        train_metrics, _ = run_epoch(model, train_loader, True, optimizer, scaler, USE_AMP)
        row = dict(epoch=epoch, train=train_metrics)
        if not final:
            row["valid"], _ = run_epoch(model, valid_loader, False, amp=USE_AMP)
        history.append(row)
        print(epoch, "train", {k: round(v, 4) for k, v in train_metrics.items()},
              "" if final else "valid {}".format({k: round(v, 4) for k, v in row["valid"].items()}))
        keep = epoch == EPOCHS if final else row["valid"]["miou"] > best
        if not final and keep: best = row["valid"]["miou"]
        if keep:
            torch.save(dict(model=model.state_dict(), config=run_config(variant, seed, final), epoch=epoch,
                            valid=None if final else row["valid"], upstream_commit=actual_commit), wpath)
    write_json(wpath.with_suffix(".json"), json_safe(dict(history=history, config=run_config(variant, seed, final),
                                                          data_tag=DATA_TAG, split_sha256=SPLIT_SHA256)))
    return wpath, round((time.time() - t0) / 60, 1)

def load_model(wpath, variant, device):
    checkpoint = torch.load(wpath, map_location=device, weights_only=True)
    model = build_model(variant, device); model.load_state_dict(checkpoint["model"]); model.eval()
    return model, checkpoint

## 6. Đánh giá trên valid
Chỉ ở `compare`. Nạp trọng số tốt nhất, đánh giá lại FP32 không tăng cường trên `valid` ở các mức sáng `BRIGHTNESS` (nhân ảnh gốc trước khi tiền xử lý; mức 1,0 là số chính). In IoU từng lớp, mIoU, precision (độ chính xác dự đoán dương) và recall (độ bao phủ) của road, ma trận nhầm lẫn ở mức 1,0, lưới ảnh | nhãn | dự đoán. Bộ dữ liệu chỉ có hai tập train và valid (giống bộ YOLO), không có tập test riêng: valid vừa dùng để chọn epoch tốt nhất nên số đo có thể hơi cao hơn so với lượt thu hoàn toàn mới. Các lượt thu của valid không xuất hiện ở train (theo `split.csv`), nhưng vẫn chỉ là một số lượt thu trên sa bàn nên chưa đại diện đầy đủ các điều kiện chạy xe.

In [ ]:
def evaluate_valid(model):
    out = []
    for b in BRIGHTNESS:
        metrics, matrix = run_epoch(model, make_loader(RoadDataset(("valid",), brightness=b), False, 0), False, amp=False)
        out.append(dict(brightness=b, metrics=metrics, confusion=matrix))
    return out

def preview_predictions(model, run_dir):
    ds = RoadDataset(("valid",)); idx = np.linspace(0, len(ds) - 1, min(PREVIEW_N, len(ds))).astype(int)
    fig, axes = plt.subplots(len(idx), 3, figsize=(12, 2.8 * len(idx)), squeeze=False)
    for r, i in enumerate(idx):
        bgr, mask = ds.raw(int(i))
        x = torch.from_numpy(preprocess_bgr(bgr)[None]).to(DEVICE)
        with torch.no_grad(): prediction = model(x).argmax(1)[0].cpu().numpy()
        for ax, img, title, cmap in zip(axes[r], (cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), mask, prediction), ("Ảnh", "Nhãn", "Dự đoán"), (None, "gray", "gray")):
            ax.imshow(img, cmap=cmap, vmin=None if cmap is None else 0, vmax=None if cmap is None else 1); ax.set_title(title); ax.axis("off")
    fig.tight_layout(); fig.savefig(run_dir / "valid_predictions.png"); plt.show(); plt.close(fig)

def plot_curves(history, run_dir):
    if not history: return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for split in ("train", "valid"):
        if split not in history[0]: continue
        axes[0].plot([h["epoch"] for h in history], [h[split]["loss"] for h in history], label=split)
        axes[1].plot([h["epoch"] for h in history], [h[split]["miou"] for h in history], label=split)
    for ax, title in zip(axes, ("Loss", "mIoU")): ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend()
    fig.tight_layout(); fig.savefig(run_dir / "training_curves.png"); plt.show(); plt.close(fig)

## 7. Xuất ONNX và đối chiếu
Xuất FP32, opset 12, không dynamic, tên input/output cố định, hình dạng 1×3×192×320 → 1×2×192×320. Không đưa chuẩn hóa hoặc softmax vào graph. So sánh ONNX Runtime với PyTorch trên tối đa `ONNX_CHECK_N` ảnh trải đều (`valid` khi compare; `train`+`valid` khi final) và input tham chiếu nếu có: sai khác tuyệt đối lớn nhất, tỉ lệ pixel argmax trùng; kiểm allclose theo dung sai `CONTRACT`. Kết quả chỉ xác nhận bản xuất, chưa xác nhận TensorRT.

`REF_DIR` chứa `NNN_input.npy` và `NNN_logits.npy`: input đã chuẩn hóa, không chuẩn hóa lại. Tỉ lệ argmax trùng với logits tham chiếu chỉ để tham khảo, không phải tiêu chí đạt hay IoU với nhãn đúng.

In [ ]:
def export_onnx(wpath, variant, run_dir, final):
    model, _ = load_model(wpath, variant, "cpu"); model = model.float()
    onnx_path = run_dir / "twinlitenetplus_192x320.onnx"
    dummy = torch.zeros(1, 3, CONTRACT["height"], CONTRACT["width"], dtype=torch.float32)
    torch.onnx.export(model, dummy, str(onnx_path), input_names=["input"], output_names=["output"],
                      opset_version=CONTRACT["opset"], dynamic_axes=None, do_constant_folding=True, dynamo=False)
    graph = onnx.load(str(onnx_path)); onnx.checker.check_model(graph)
    assert next(o.version for o in graph.opset_import if o.domain == "") == 12
    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    assert session.get_inputs()[0].name == "input" and session.get_inputs()[0].shape == [1, 3, 192, 320]
    assert session.get_outputs()[0].name == "output" and session.get_outputs()[0].shape == [1, 2, 192, 320]
    stats = dict(max_error=0.0, matched=0, pixels=0, count=0)
    def compare_export(x):
        assert x.shape == (1, 3, 192, 320) and x.dtype == np.float32 and np.isfinite(x).all()
        with torch.no_grad(): pt = model(torch.from_numpy(x)).numpy()
        exported = session.run(["output"], {"input": x})[0]
        assert np.isfinite(pt).all() and np.isfinite(exported).all()
        stats["max_error"] = max(stats["max_error"], float(np.max(np.abs(exported - pt))))
        stats["matched"] += int(np.sum(exported.argmax(1) == pt.argmax(1))); stats["pixels"] += int(pt.shape[0] * pt.shape[2] * pt.shape[3]); stats["count"] += 1
        np.testing.assert_allclose(exported, pt, atol=CONTRACT["ort_atol"], rtol=CONTRACT["ort_rtol"])
        return pt
    ds = RoadDataset(("train", "valid") if final else ("valid",))
    for i in np.linspace(0, len(ds) - 1, min(ONNX_CHECK_N, len(ds))).astype(int):
        bgr, _ = ds.raw(int(i)); compare_export(preprocess_bgr(bgr)[None])
    reference = []
    if REF_DIR:
        files = sorted(Path(REF_DIR).glob("*_input.npy")); assert files, "REF_DIR không có tensor input"
        for path in files:
            x = np.load(path, allow_pickle=False)
            old = np.load(path.with_name(path.name.replace("_input.npy", "_logits.npy")), allow_pickle=False)
            if old.shape == (2, 192, 320): old = old[None]
            assert old.shape == (1, 2, 192, 320) and np.isfinite(old).all()
            reference.append(dict(input=path.name, argmax_match_old=float(np.mean(compare_export(x).argmax(1) == old.argmax(1)))))
    comparison = dict(max_absolute_error=stats["max_error"], argmax_match=stats["matched"] / stats["pixels"], inputs=stats["count"])
    print("  ONNX Runtime / PyTorch:", comparison)
    info = dict(task="segmentation", architecture="twinlitenetplus", class_names=["background", "road"],
                num_classes=2, mask_values=dict(background=0, road=1), source_image_hw=[720, 1280],
                crop_y=[0, 720], crop_x=[0, 1280], resize="cv2.resize INTER_LINEAR (full -> input)",
                input_name="input", input_shape=[1, 3, 192, 320], input_hw=[192, 320], input_dtype="float32",
                color_order="RGB", layout="NCHW", normalization=dict(scale=CONTRACT["scale"], mean=CONTRACT["mean"], std=CONTRACT["std"]),
                imagenet_normalization_in_model=False, preprocessing_in_model=False, output_name="output",
                output_shape=[1, 2, 192, 320], output_type="logits", activation="softmax_over_channels_argmax",
                road_rule="output[0,1] > output[0,0]", dynamic=False, fake_data=FAKE_DATA, opset=12,
                onnx_sha256=hashlib.sha256(onnx_path.read_bytes()).hexdigest())
    write_json(run_dir / "twinlitenetplus_192x320_road_info.json", info)
    return comparison, reference

## 8. Chạy `compare` hoặc `final`
`run_key = DATA_TAG|biến thể|seed`. Ở `compare`, lần chạy đã có trong `twinlite_compare.csv` được bỏ qua. Mỗi lần chạy ghi `twinlite_metrics.json` (cấu hình, commit nguồn, phiên bản gói, số ảnh từng tập, chỉ số theo mức sáng, lịch sử train, so sánh ONNX, SHA256 của `split.csv`) để lần sau đối chiếu. **Số đo thật chỉ ghi sau khi chạy xong;** model `final` không có số đo.

In [ ]:
SUMMARY = []
final = MODE == "final"
names = VARIANTS if MODE == "compare" else [FINAL_VARIANT]
seeds = SEEDS if MODE == "compare" else [FINAL_SEED]
skip = done_keys(COMPARE_CSV) if MODE == "compare" else set()
for v in names:
    for k, seed in enumerate(seeds):
        key = "{}|{}|{}".format(DATA_TAG, v, seed)
        if key in skip: print("Bỏ qua (đã có):", key); continue
        print(v, "seed", seed, "| chế độ", MODE)
        wpath, minutes = train_or_load(v, seed, final)
        name = "{}_s{}{}".format(v, seed, "_final" if final else "")
        run_dir = OUT / name; run_dir.mkdir()
        model, checkpoint = load_model(wpath, v, DEVICE)
        shutil.copy2(wpath, run_dir / ("last.pt" if final else "best.pt"))
        hist_path = wpath.with_suffix(".json")
        history = json.loads(hist_path.read_text(encoding="utf-8"))["history"] if hist_path.exists() else []
        plot_curves(history, run_dir)
        params = int(sum(p.numel() for p in model.parameters()))
        sweep, confusion = None, None
        if not final:
            sweep = evaluate_valid(model)
            base = dict(run_key=key, run_id=RUN_ID, data_tag=DATA_TAG, variant=v, seed=seed, epochs=EPOCHS, batch_size=BATCH, lr=LR,
                        params=params, train_min=minutes, n_train=SPLIT_COUNTS["train"], n_valid=SPLIT_COUNTS["valid"],
                        best_epoch=checkpoint["epoch"], weights=str(wpath.relative_to(ROOT)), split_sha256=SPLIT_SHA256,
                        upstream_commit=actual_commit)
            rows = [dict(base, brightness=s["brightness"], **{k: x for k, x in s["metrics"].items() if k != "loss"}) for s in sweep]
            append_csv(COMPARE_CSV, rows); SUMMARY += rows
            main = next(s for s in sweep if s["brightness"] == 1.0)
            confusion = main["confusion"]
            print("  valid (FP32, sáng 1,0):", {a: round(b, 4) for a, b in main["metrics"].items()},
                  "| ma trận nhầm lẫn (hàng=nhãn, cột=dự đoán):", confusion.tolist(), "| epoch tốt nhất", checkpoint["epoch"])
            if k == 0 and PREVIEW_N: preview_predictions(model, run_dir)
        comparison, reference = None, []
        if final or (EXPORT_FIRST_SEED and k == 0):
            comparison, reference = export_onnx(wpath, v, run_dir, final)
        metrics = dict(mode=MODE, run_key=key, config=run_config(v, seed, final), upstream_commit=actual_commit, packages=PACKAGES,
                       trained_on="train+valid" if final else "train", best_epoch=None if final else checkpoint["epoch"],
                       best_valid=None if final else checkpoint["valid"], split_images=SPLIT_COUNTS,
                       valid_fp32=None if final else next(s for s in sweep if s["brightness"] == 1.0)["metrics"],
                       valid_by_brightness=None if final else {str(s["brightness"]): s["metrics"] for s in sweep},
                       confusion_matrix=None if confusion is None else confusion.tolist(), history=history,
                       onnx_comparison=comparison, old_model_reference=reference, params=params,
                       split_source=dict(file=SPLIT_CSV if not FAKE_DATA else "fake", sha256=SPLIT_SHA256, zip=zip_path.name),
                       note="final: train trên toàn bộ ảnh có nhãn, không có chỉ số" if final else "valid dùng chọn epoch nên số đo hơi lạc quan")
        write_json(run_dir / "twinlite_metrics.json", json_safe(metrics))
print("Xong. Bảng so sánh:", COMPARE_CSV if MODE == "compare" else "(final không đánh giá)")

## Tóm tắt so sánh
Chỉ ở `compare`: mỗi dòng một biến thể ở mức sáng 1,0, kèm trung bình 5 mức sáng. Đọc `iou_road` và `recall_road` (bỏ sót mặt đường làm xe tưởng hết đường) trước, rồi `precision_road` (nhận nhầm vùng ngoài đường là đường làm xe đi lệch). Số đo chỉ là valid; tốc độ thật phải đo `trtexec` trên xe (`params` chỉ để tham khảo).

In [ ]:
if MODE == "compare" and COMPARE_CSV.exists():
    import pandas as pd
    try: from IPython.display import display
    except ImportError: display = print
    df = pd.read_csv(COMPARE_CSV, encoding="utf-8-sig"); df = df[df["data_tag"] == DATA_TAG]
    cols = ["iou_road", "iou_background", "miou", "precision_road", "recall_road"]
    for title, part in (("Mức sáng 1,0", df[df["brightness"] == 1.0]), ("Trung bình 5 mức sáng", df)):
        print(title); display(part.groupby(["variant", "seed"])[cols].mean().groupby("variant").agg(["mean", "std"]).round(4))
    print(df.groupby("variant")[["params", "train_min", "best_epoch", "n_train", "n_valid"]].first())
    shutil.copy2(COMPARE_CSV, OUT / COMPARE_CSV.name)

## 9. Lưu kết quả
`results.zip` = cả thư mục `<RUN_ID>` của lần chạy: một thư mục con cho mỗi lần chạy `<biến thể>_s<seed>[_final]/` (checkpoint, ONNX, `_road_info.json`, `twinlite_metrics.json`, biểu đồ) cùng `data_preview.png` và bản sao CSV so sánh. Tải `results.zip` trước khi kết thúc runtime (trọng số cũng ở lại trên Drive). Không đưa `best.pt`, `last.pt`, `twinlite_metrics.json` vào repo GitHub; chỉ chép ONNX và `_road_info.json` vào `models/` sau khi đối chiếu.

In [ ]:
archive_path = shutil.make_archive(str(WORK / "results"), "zip", OUT)
print("results.zip:", archive_path, "|", Path(archive_path).stat().st_size, "byte")
if IN_COLAB:
    from google.colab import files
    files.download(archive_path)

## 10. Hạn chế
- Phải điền commit nguồn đã kiểm tra; ô dò API sẽ dừng nếu repo tại commit đó khác API đã mô tả.
- Chia tập theo `split.csv` (hai tập train, valid, giống bộ YOLO): ảnh có số lượt trong tên được nhóm theo series và số lượt. Ảnh `park_` và `road_them_` được nhóm theo 10 số thứ tự liền nhau; cả 8 ảnh `image<số>` là một nhóm, cả 7 ảnh `h9342_<số>` là một nhóm. Mỗi nhóm chỉ thuộc một tập; các khung gần nhau khác nhóm vẫn có nguy cơ nằm ở hai tập. IoU đo trên cách chia theo lượt thu nên không so trực tiếp được với số đo trên cách chia ngẫu nhiên theo ảnh hoặc với số đo trên tập test của bản chia ba tập cũ.
- Không có tập test riêng; epoch tốt nhất được chọn theo valid nên chỉ số valid có thể hơi lạc quan. Model `final` (train+valid) không có chỉ số nào để báo.
- Seed cố định giúp lặp lại nhưng khác GPU/CUDA hoặc khác phiên bản torch/numpy (không ghim, theo Colab lúc chạy) có thể cho kết quả khác. Mã chưa kiểm với mọi phiên bản torch; nếu lỗi API (xuất ONNX, GradScaler) thì ghi lại thông báo lỗi và phiên bản đã in ở mục 2.
- Kiểm ONNX Runtime không thay thế build TensorRT 8.2 và đối chiếu trên Jetson, cũng không xác nhận độ an toàn hoặc chất lượng bám đường trên sa bàn.